# 02 - Exploratory Data Analysis (PuneBite Analyzer)

**Objective:** understand the cleaned data before modelling: distributions, localities, restaurant types, price, correlations, and review patterns.
**Input:** `data/processed/restaurants_clean.csv`
**Output:** insights and charts for the report (no file saved)
**Syllabus link:** Unit II (EDA: summary statistics, distributions, correlation and covariance, grouping, aggregation, pivot tables); Lab 2.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 200)
sns.set_theme(style="whitegrid")

df = pd.read_csv("../data/processed/restaurants_clean.csv")
rated = df[df["has_rating"]].copy()          # only restaurants with a real rating

# amenity columns sit between star1_pct and spam_review_count in the file
cols = list(df.columns)
amenity_cols = cols[cols.index("star1_pct") + 1 : cols.index("spam_review_count")]

print("All restaurants:", df.shape, "| Rated:", rated.shape, "| Amenities:", len(amenity_cols))
df.head(3)

## 1. Summary statistics

In [ ]:
num_cols = ["rating", "votes", "cost_capped", "n_cuisines", "n_amenities"]
summary = df[num_cols].agg(["count", "mean", "median", "std", "var", "min", "max"]).T
summary["skew"] = df[num_cols].skew()
summary["kurtosis"] = df[num_cols].kurt()
summary.round(2)

`votes` has a mean far above its median, a classic sign of a **right-skewed** distribution: a few very popular restaurants and a huge number of barely-reviewed ones.

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(15, 4))
sns.histplot(rated["rating"], bins=20, kde=True, ax=ax[0]);            ax[0].set_title("Rating")
sns.histplot(np.log1p(df["votes"]), bins=30, ax=ax[1]);                ax[1].set_title("Votes (log scale)")
sns.histplot(df["cost_capped"], bins=30, ax=ax[2]);                    ax[2].set_title("Cost for two (capped)")
plt.tight_layout(); plt.show()

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(15, 3.2))
sns.boxplot(x=rated["rating"], ax=ax[0]);        ax[0].set_title("Rating")
sns.boxplot(x=df["votes"], ax=ax[1]);            ax[1].set_title("Votes")
sns.boxplot(x=df["cost_capped"], ax=ax[2]);      ax[2].set_title("Cost for two")
plt.tight_layout(); plt.show()

## 2. Who has no rating?

In [ ]:
pd.crosstab(df["has_rating"], df["votes"] > 0, rownames=["has rating"], colnames=["has votes"])

Almost every unrated restaurant has zero votes, so "no rating" simply means "nobody has reviewed it yet". These rows are kept for locality counts but excluded from rating analysis and models.

## 3. Localities (grouping and aggregation)

In [ ]:
loc_counts = df["locality"].value_counts()
print("Localities:", len(loc_counts), "| with >= 40 rated restaurants:", (rated["locality"].value_counts() >= 40).sum())

plt.figure(figsize=(9, 5))
sns.barplot(x=loc_counts.head(15).values, y=loc_counts.head(15).index, color="steelblue")
plt.title("Top 15 localities by number of restaurants"); plt.xlabel("Restaurants"); plt.show()

In [ ]:
# Aggregation per locality (only localities with >= 40 rated restaurants)
loc_pivot = (rated.groupby("locality")
                  .agg(rated_count=("rating", "count"),
                       avg_rating=("rating", "mean"),
                       median_cost=("cost_capped", "median"),
                       median_votes=("votes", "median")))
loc_pivot = loc_pivot[loc_pivot["rated_count"] >= 40].sort_values("avg_rating", ascending=False)
print(len(loc_pivot), "localities")
loc_pivot.head(8).round(2)

In [ ]:
loc_pivot.tail(8).round(2)

In [ ]:
top_bottom = pd.concat([loc_pivot.head(8), loc_pivot.tail(8)])
plt.figure(figsize=(9, 6))
sns.barplot(x=top_bottom["avg_rating"], y=top_bottom.index, color="seagreen")
plt.axvline(rated["rating"].mean(), color="red", linestyle="--", label="Pune average")
plt.xlim(3, None); plt.legend(); plt.title("Best and worst rated localities (>= 40 rated restaurants)")
plt.xlabel("Average rating"); plt.show()

In [ ]:
# Locality x restaurant type: how many of each type in the 10 biggest localities
top_loc = loc_counts.head(10).index
top_types = df["primary_type"].value_counts().head(6).index
ct = pd.crosstab(df[df["locality"].isin(top_loc)]["locality"],
                 df[df["locality"].isin(top_loc)]["primary_type"])[top_types]
plt.figure(figsize=(9, 5))
sns.heatmap(ct, annot=True, fmt="d", cmap="YlGnBu")
plt.title("Restaurant types in the 10 biggest localities"); plt.show()

In [ ]:
# A true pivot table: average rating for each locality (rows) x restaurant type (columns)
big_loc = loc_pivot.sort_values("rated_count", ascending=False).head(10).index
big_types = rated["primary_type"].value_counts().head(5).index
pt = pd.pivot_table(rated[rated["locality"].isin(big_loc) & rated["primary_type"].isin(big_types)],
                    index="locality", columns="primary_type", values="rating", aggfunc="mean")
pt.round(2)

## 4. Restaurant type and cuisine

In [ ]:
type_pivot = (rated.groupby("primary_type")
                   .agg(rated_count=("rating", "count"),
                        avg_rating=("rating", "mean"),
                        median_cost=("cost_capped", "median")))
type_pivot = type_pivot[type_pivot["rated_count"] >= 40].sort_values("avg_rating", ascending=False)
type_pivot.round(2)

In [ ]:
# One restaurant has several cuisines, so 'explode' the list into one row per cuisine
cuis = rated.assign(cuisine=rated["cuisines"].str.split("|")).explode("cuisine")
cuis_pivot = cuis.groupby("cuisine")["rating"].agg(["count", "mean"])
cuis_pivot = cuis_pivot[cuis_pivot["count"] >= 100].sort_values("mean", ascending=False)

fig, ax = plt.subplots(1, 2, figsize=(14, 5))
sns.barplot(x=df["cuisines"].str.split("|").explode().value_counts().head(12).values,
            y=df["cuisines"].str.split("|").explode().value_counts().head(12).index, color="steelblue", ax=ax[0])
ax[0].set_title("Most common cuisines (all restaurants)")
sns.barplot(x=cuis_pivot["mean"].head(12), y=cuis_pivot.head(12).index, color="seagreen", ax=ax[1])
ax[1].set_xlim(3, None); ax[1].set_title("Highest rated cuisines (>= 100 rated)")
plt.tight_layout(); plt.show()

## 5. Price and rating

In [ ]:
rated["price_band"] = pd.qcut(rated["cost_capped"], 5, duplicates="drop")
band = rated.groupby("price_band", observed=True)["rating"].agg(["count", "mean", "median"])
print(band.round(2))

plt.figure(figsize=(8, 4))
sns.boxplot(data=rated, x="price_band", y="rating")
plt.xticks(rotation=20); plt.title("Rating by price band"); plt.show()

## 6. Correlation and covariance

In [ ]:
corr_cols = ["rating", "votes", "cost_capped", "n_amenities", "n_cuisines"]
fig, ax = plt.subplots(1, 2, figsize=(13, 4.5))
sns.heatmap(rated[corr_cols].corr(method="pearson"), annot=True, fmt=".2f", cmap="coolwarm", vmin=-1, vmax=1, ax=ax[0])
ax[0].set_title("Pearson correlation")
sns.heatmap(rated[corr_cols].corr(method="spearman"), annot=True, fmt=".2f", cmap="coolwarm", vmin=-1, vmax=1, ax=ax[1])
ax[1].set_title("Spearman (rank) correlation")
plt.tight_layout(); plt.show()

In [ ]:
# Covariance depends on the units of each variable, so it is hard to compare across pairs.
# Correlation is covariance rescaled to the range -1..+1.
rated[corr_cols].cov().round(2)

**Why two correlations?** Pearson measures straight-line relationships and is distorted by the skewed `votes`. Spearman uses ranks, so it is more reliable for skewed data. Their gap for `rating` vs `votes` shows the relationship is strong but not a straight line.

## 7. Votes and rating reliability

In [ ]:
vote_bins = pd.cut(rated["votes"], [-1, 4, 9, 24, 49, 99, 499, 100000],
                   labels=["0-4", "5-9", "10-24", "25-49", "50-99", "100-499", "500+"])
vb = rated.groupby(vote_bins, observed=True)["rating"].agg(["count", "mean", "std"])
print(vb.round(2))

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
sns.barplot(x=vb.index.astype(str), y=vb["mean"], color="steelblue", ax=ax[0])
ax[0].set_title("Average rating by number of votes"); ax[0].set_ylim(2.5, None)
sns.boxplot(data=rated.assign(vote_bin=vote_bins), x="vote_bin", y="rating", ax=ax[1])
ax[1].set_title("Rating spread by votes")
plt.tight_layout(); plt.show()

Restaurants with very few votes cluster tightly around a low rating, while heavily reviewed ones rate much higher. This means **a rating is not equally trustworthy for every restaurant**, which is the motivation for our *Trust-adjusted Rating* in the Hidden Gems feature. It also warns us that `votes` is a very strong predictor, partly because popularity and rating move together.

## 8. Review patterns (star percentages)

In [ ]:
star_cols = ["star5_pct", "star4_pct", "star3_pct", "star2_pct", "star1_pct"]
rated["rating_band"] = pd.cut(rated["rating"], [1.9, 3.0, 3.5, 4.0, 5.0],
                              labels=["<=3.0", "3.0-3.5", "3.5-4.0", ">4.0"])
mix = rated.groupby("rating_band", observed=True)[star_cols + ["polarization_pct"]].mean().round(1)
mix

In [ ]:
mix[star_cols].plot(kind="bar", stacked=True, figsize=(8, 4), colormap="RdYlGn_r")
plt.title("Average review mix by rating band"); plt.ylabel("% of reviews"); plt.xticks(rotation=0)
plt.legend(title=None, bbox_to_anchor=(1.02, 1), loc="upper left"); plt.tight_layout(); plt.show()

In [ ]:
# 'Love it or hate it' restaurants: many 5-star AND many 1-star reviews (needs enough votes to mean something)
polar = (rated[rated["votes"] >= 100]
         .sort_values("polarization_pct", ascending=False)
         [["name", "locality", "rating", "votes", "star5_pct", "star1_pct", "polarization_pct"]].head(10))
polar

## 9. Which amenities go with higher ratings? (preview of Unit III tests)

This is a first look only. Whether a difference is *statistically significant* is decided by the t-tests in notebook 03.

In [ ]:
rows = []
for a in amenity_cols:
    with_a = rated.loc[rated[a] == 1, "rating"]
    without = rated.loc[rated[a] == 0, "rating"]
    if len(with_a) >= 100 and len(without) >= 100:
        rows.append((a, len(with_a), with_a.mean(), without.mean(), with_a.mean() - without.mean()))
am = (pd.DataFrame(rows, columns=["amenity", "n_with", "mean_with", "mean_without", "difference"])
        .sort_values("difference", ascending=False))
am.head(8).round(2)

In [ ]:
plt.figure(figsize=(8, 6))
pick = pd.concat([am.head(8), am.tail(5)])
sns.barplot(data=pick, x="difference", y="amenity", color="steelblue")
plt.axvline(0, color="black", linewidth=0.8)
plt.title("Rating difference: restaurants with vs without the amenity"); plt.show()

## 10. Key findings (check against your own outputs and put in the report)

1. **Skewed popularity:** half of all restaurants have fewer than ~10 votes while a few have thousands, so `votes` needs a log transform or ranks for modelling.
2. **Ratings sit in a narrow band:** the median is 3.4 and only about 11% of rated restaurants reach 4.0, so the "Highly Rated" class will be small.
3. **Votes and rating move together** (Spearman about 0.7): low-vote restaurants have unreliable, low ratings. This justifies a trust-adjusted rating.
4. **Locality matters:** Koregaon Park, Kalyani Nagar, FC Road and Bund Garden Road are at the top, while areas like Bhosari, Dhanori and Vishrantwadi are at the bottom (among localities with at least 40 rated restaurants).
5. **Type and price matter:** bars, lounges, dessert parlours and cafes beat quick-bite and takeaway places, and the most expensive price band stands out.
6. **Amenities that signal premium service** (live sports screening, valet parking, live music, table booking recommended) show clearly higher average ratings.
7. **Correlation is not causation:** a bar having a higher rating does not mean adding a bar raises a restaurant's rating.

**Next:** `03_statistics.ipynb` (t-test, ANOVA, chi-square).